# 02-local-news-topics

See [project guide](../../README.md) and [data requirements](../../data/README.md) before execution. Workspace: `data/news/`. External inputs are not included. Run cells in order; model fitting and network collection are not run during repository checks.


In [ ]:
from pathlib import Path
import sys
import os
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'project_paths.py').is_file())
sys.path.insert(0, str(PROJECT_ROOT))
from project_paths import workspace
os.chdir(workspace('news'))


In [ ]:
%pprint

In [ ]:
# Imported Libraries
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as img
from matplotlib.pyplot import MultipleLocator
from IPython.display import display ### 列表展示库 display(df)

import os
import re # 正则库
import time
import datetime

import threading # 多线程
import urllib.request # 爬虫 request 库

from concurrent.futures import ThreadPoolExecutor , as_completed # 线程池
from datetime import datetime # datetime.datetime
from scipy import stats # 统计
from tqdm import tqdm   # 进度条

import seaborn as sns


# nltk
import nltk

# nltk.download('punkt')                # run when first time use this library
# nltk.download('stopwords')            # 用于分词

# nltk.download('omw-1.4')              # run when first time use this library
# nltk.download('wordnet')              # 用于词性还原

# use stopwords.words() to obtain stopwords list
from nltk.corpus import stopwords as stpw     
# words extraction, E.G. word_tokenize('an apple') => ['an', 'apple']
from nltk.tokenize import word_tokenize
# WordNetLemmatizer().lemmatize('running') => 'run'
# WordNetLemmatizer().lemmatize('bats') => 'bat'
# Only useful with lower capital words !!!
from nltk.stem import WordNetLemmatizer 

# sklearn
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import GridSearchCV

# gensim
import gensim
import gensim.corpora as corpora
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel, LdaModel

In [ ]:
# Locate folder
os.chdir(workspace('news'))
path = workspace('news')      # Change path here

# wpxi = pd.read_csv(str(path) + 'wpxi.csv')
cbs  = pd.read_csv(str(path) + 'cbs_Pittsburgh.csv')
# wtae  = pd.read_csv(str(path) + 'wtae.csv')
# ppg  = pd.read_csv(str(path) + 'PittsburghPostGazette.csv')
# ppg_covid  = pd.read_csv(str(path) + 'PittsburghPostGazette_covid-page.csv')

save_data = True
save_plot = True

In [ ]:
cbs.head()

# Data Cleaning

In [ ]:
# Import stopwords

stopwords = []
htmlcodes = []
punctuations = []

with open("stopwords_en.txt", "r") as f1:
    for stopword in f1.readlines():
        stopwords.append(stopword.strip('\n'))

with open("stopwords_html.txt", "r") as f2:
    for htmlcode in f2.readlines():
        htmlcodes.append(htmlcode.strip('\n'))     
        
with open("stopwords_punctuation.txt", "r") as f3:
    for punctuation in f3.readlines():
        punctuations.append(punctuation.strip('\n'))        
        
stopwords  = list( set( stopwords + htmlcodes + punctuations ) )
htlm_puncs = list( set( htmlcodes + punctuations ) )


In [ ]:
# Step 1 - Dataframe cleaning : drop null and duplicates

def clean_df(df):
    required = ['Date', 'Title', 'Keywords', 'Content']
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f'Missing news columns: {sorted(missing)}')
    df = df.dropna(subset=['Date', 'Title']).drop_duplicates('Title').copy()
    df[['Keywords', 'Content']] = df[['Keywords', 'Content']].fillna('')
    df['Date'] = pd.to_datetime(df['Date'], errors='raise').dt.strftime('%Y/%m/%d')
    return df.sort_values('Date').reset_index(drop=True)


In [ ]:
# Step 2 - Content Cleaning : Delet stopwords and punctuations

def cleanContent( text , addtional_stopwords ):
    
    '''     
        import nltk
        # nltk.download('punkt')                # run when first time use this library
        # nltk.download('stopwords')            # run when first time use the following module
        # nltk.download('wordnet')              # run when first time use the following module

        from nltk.corpus import stopwords sas stpw     # use stopwords.words() to obtain stopwords list
        from nltk.tokenize import word_tokenize        # words extraction
       
    '''
    text = text.lower() # to lower case
    
    for html_punc in htlm_puncs :
        text = text.replace( html_punc , ' ' )
        
    text = text.replace('hong kong' , 'hongkong')
    text = text.replace('wu han' , 'wuhan') 
    text = text.replace('dept' , 'department')
    text = text.replace('tom wolf' , 'tomwolf')
    text = text.replace('reopening' , 'reopen')
    text = text.replace('closed' , 'close')
    text = text.replace('closure' , 'close')
    text = text.replace('closing' , 'close')
    text = text.replace('governor' , 'government')
    
    text = re.sub(r'<(.*?)>', '', text)
    text = re.sub(r'_+', '', text)
    text = re.sub(r'\s\s+', ' ', text)
    text = re.sub(r'[0-9]', '', text)
    
    words = word_tokenize(text) # word extraction
    stopwords = list ( set( addtional_stopwords + stpw.words('english') ) )
    
    # WordNetLemmatizer().lemmatize(<world>) only useful with lower capital words !!!
    taken_words = [ WordNetLemmatizer().lemmatize(word).title() for word in words if word not in stopwords ]
    
    clean_text = (" ").join( taken_words )
    
    return clean_text


def cleanContents( texts_list ):
    
    clean_contents = []
    
    if __name__ == "__main__" :
        with ThreadPoolExecutor() as pool :
            clean_contents_generator = pool.map( lambda content : cleanContent( content , stopwords ) , texts_list )
            for clean_content in tqdm(clean_contents_generator):
                clean_contents.append(clean_content)
    
    return clean_contents

In [ ]:
WordNetLemmatizer().lemmatize('closing').title()

In [ ]:
# Clean data in the dataframe
cbs = clean_df( cbs )

In [ ]:
# Clean the contents for the contents of the news
clean_titles = cleanContents( list( cbs['Title'] ) )
clean_keywords = cleanContents( list( cbs['Keywords'] ) )
clean_title_keywords = [(clean_titles[i] + ' ' + clean_keywords[i]) 
                        for i in range(len(clean_titles))]
clean_contents = cleanContents( list( cbs['Content'] ) )

In [ ]:
cbs['Title'] = clean_titles
cbs['Keywords'] = clean_keywords
cbs['Content'] = clean_contents
cbs['Selected_Words'] = clean_title_keywords
days = cbs.Date
display(cbs)

In [ ]:
# Selected words by date
def sumstr(df):
    return ' '.join(list(df.iloc[:,1]))

# Notes
# Use this to display groupby : df.groupby([GroupBy]).apply(lambda a: a[:]) ) 
# groupby( [GroupBy] ).apply( func ) & groupby( [GroupBy] ).func()  
# carries out a group-wise calculation, the first argument of func is a dataframe for each group
# returns as a dataframe or a series

cbs_list1 = cbs.groupby('Date')['Title'].agg(' '.join)
cbs_list2 = cbs.groupby('Date')['Keywords'].agg(' '.join)
cbs_list3 = cbs.groupby('Date')['Content'].agg(' '.join)
cbs_list4 = cbs.groupby('Date')['Selected_Words'].agg(' '.join)

cbs_list1 = pd.DataFrame(cbs_list1, columns = ['Title'] )
cbs_list2 = pd.DataFrame(cbs_list2, columns = ['Keywords'] )
cbs_list3 = pd.DataFrame(cbs_list3, columns = ['Content'] )
cbs_list4 = pd.DataFrame(cbs_list4, columns = ['Selected_Words'] )

display(cbs_list1)

In [ ]:
# Save the Data
if save_data :
    cbs.to_csv('./Data/CBS_KDKA/cbs_clean.csv', encoding = "utf_8_sig" ,  index = False)
    cbs_list1.to_csv('./Data/CBS_KDKA/cbs_title.csv', encoding = "utf_8_sig" ,  index = True)
    cbs_list2.to_csv('./Data/CBS_KDKA/cbs_keywords.csv', encoding = "utf_8_sig" ,  index = True)
    cbs_list3.to_csv('./Data/CBS_KDKA/cbs_content.csv', encoding = "utf_8_sig" ,  index = True)
    cbs_list4.to_csv('./Data/CBS_KDKA/cbs_selectedwords.csv', encoding = "utf_8_sig" ,  index = True)

# Whole Period Word Counting

In [ ]:
# Read Data
cbs       = pd.read_csv('./Data/CBS_KDKA/cbs_clean.csv', encoding = "utf_8_sig")
cbs_list1 = pd.read_csv('./Data/CBS_KDKA/cbs_title.csv', encoding = "utf_8_sig")
cbs_list2 = pd.read_csv('./Data/CBS_KDKA/cbs_keywords.csv', encoding = "utf_8_sig")
cbs_list3 = pd.read_csv('./Data/CBS_KDKA/cbs_content.csv', encoding = "utf_8_sig")
cbs_list4 = pd.read_csv('./Data/CBS_KDKA/cbs_selectedwords.csv', encoding = "utf_8_sig")


cbs_list1.set_index('Date', inplace = True) # Title
cbs_list2.set_index('Date', inplace = True) # Keywords
cbs_list3.set_index('Date', inplace = True) # Content
cbs_list4.set_index('Date', inplace = True) # Selectedwords

# Selected words for the whole period
cbs_all_words1 = ' '.join(list(cbs_list1.iloc[:,0])).split()
cbs_all_words2 = ' '.join(list(cbs_list2.iloc[:,0])).split()
cbs_all_words3 = ' '.join(list(cbs_list3.iloc[:,0])).split()
cbs_all_words4 = ' '.join(list(cbs_list4.iloc[:,0])).split()

days = cbs_list1.index

In [ ]:
# Define word count function
def count_words( wordslst ):
    df = pd.DataFrame( {'Words': wordslst, 'Count':np.zeros(len(wordslst))} )
    wordcounts = df.groupby('Words').agg({'Count':np.size}).sort_values(by = 'Count', ascending = False)
    wordcounts = wordcounts.reset_index()
    
    # sort_values(by=‘ColumnName’, axis=0 , ascending = True , inplace = False, na_position = ‘last’ )
    # the column name of agg must be pre-determined
    # df = df.reset_index() can maintain the original index as a column and add a new index starting from 0

    return wordcounts

def checkWords( word , wordcounts ):
    # wordcounts is the df defined from above using count_words( wordslst )     
    return wordcounts[ wordcounts['Word'] == word ]


In [ ]:
cbs_total_wordcounts1 = count_words( cbs_all_words1 )
cbs_total_wordcounts2 = count_words( cbs_all_words2 )
cbs_total_wordcounts3 = count_words( cbs_all_words3 )
cbs_total_wordcounts4 = count_words( cbs_all_words4 )

print(len(cbs_total_wordcounts1))
print(len(cbs_total_wordcounts2))
print(len(cbs_total_wordcounts3))
print(len(cbs_total_wordcounts4))

display(cbs_total_wordcounts1.iloc[:20].T)
display(cbs_total_wordcounts2.iloc[:20].T)
display(cbs_total_wordcounts3.iloc[:20].T)
display(cbs_total_wordcounts4.iloc[:20].T)

# Example
display(checkWords( 'Open' , cbs_total_wordcounts1 ))
display(checkWords( 'Open'   , cbs_total_wordcounts2 ))
display(checkWords( 'Open'   , cbs_total_wordcounts3 ))
display(checkWords( 'Open'   , cbs_total_wordcounts4 ))

In [ ]:
# Generate wordcloud
# Import the wordcloud library
# import matplotlib.pyplot as plt

from wordcloud import WordCloud
from PIL import Image

def drawWordCloud( wordcounts , topNumber , figuresize , filename ): 
    word_frequency = {x[0]:x[1] for x in wordcounts.head(topNumber).values} # 返回二维列表，外层元素为Observation 
    
    if len(filename) == 0 :
        wordcloud = WordCloud(scale=10, background_color="white", 
                              max_words=topNumber, contour_width=3, contour_color='steelblue')
    # scale - pic quality
    else :
        shape = np.array( Image.open(os.path.join(os.getcwd(),filename)) ) # current working path os.getcwd()
        wordcloud = WordCloud(scale=10, background_color="white", 
                              mask=shape, max_words=topNumber)
     
    # Generate a word cloud
    wordcloud.fit_words(word_frequency)
    
    # Visualize the word cloud
    fig = plt.figure( figsize = figuresize ) 
    plt.imshow(wordcloud)
    plt.axis("off")
    plt.show()
    
    return fig

In [ ]:
fig1 = drawWordCloud( cbs_total_wordcounts1 , 500 , (10,40) , 'covid.png' ) # 必须为白底图案

In [ ]:
fig2 = drawWordCloud( cbs_total_wordcounts2 , 200 , (10,10) , '' ) 

In [ ]:
fig3 = drawWordCloud( cbs_total_wordcounts3 , 500 , (10,40) , 'covid.png' )

In [ ]:
fig4 = drawWordCloud( cbs_total_wordcounts4 , 500 , (10,40) , 'covid.png' ) 

In [ ]:
if save_plot :
        fig3.savefig(r'./Plot/CBS_KDKA/cbs_wordcloud3.jpg', transparent = False, bbox_inches = 'tight')

In [ ]:
# Save the Data
if save_data :
    cbs_total_wordcounts1.to_csv('./Data/CBS_KDKA/cbs_total_wordcounts_title.csv', encoding = "utf_8_sig" ,  index = False)
    cbs_total_wordcounts2.to_csv('./Data/CBS_KDKA/cbs_total_wordcounts_keywords.csv', encoding = "utf_8_sig" ,  index = False)
    cbs_total_wordcounts3.to_csv('./Data/CBS_KDKA/cbs_total_wordcounts_content.csv', encoding = "utf_8_sig" ,  index = False)
    cbs_total_wordcounts4.to_csv('./Data/CBS_KDKA/cbs_total_wordcounts_selectwords.csv', encoding = "utf_8_sig" ,  index = False)

# TF-IDF Analysis

## Data Processing

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Read Data
cbs       = pd.read_csv('./Data/CBS_KDKA/cbs_clean.csv', encoding = "utf_8_sig")
cbs_list1 = pd.read_csv('./Data/CBS_KDKA/cbs_title.csv', encoding = "utf_8_sig")
cbs_list2 = pd.read_csv('./Data/CBS_KDKA/cbs_keywords.csv', encoding = "utf_8_sig")
cbs_list3 = pd.read_csv('./Data/CBS_KDKA/cbs_content.csv', encoding = "utf_8_sig")
cbs_list4 = pd.read_csv('./Data/CBS_KDKA/cbs_selectedwords.csv', encoding = "utf_8_sig")

cbs_list1.set_index('Date', inplace = True) # Title
cbs_list2.set_index('Date', inplace = True) # Keywords
cbs_list3.set_index('Date', inplace = True) # Content
cbs_list4.set_index('Date', inplace = True) # Selectedwords

days = list(cbs_list1.index)

In [ ]:
# Notes

# from sklearn.feature_extraction.text import TfidfVectorizer

# step 1 define the document list
corpus = ['This is the first document.',
          'This document is the second document.',
          'And this is the third one.',
          'Is this the first document?']

# step 2 construct the model object
vectorizer = TfidfVectorizer() 

# step 3 fit the model with the document sequence as the form of "corpus" 
X = vectorizer.fit_transform(corpus) 

# step 4 get terms appeared in the whole documnet set in the form of an array
terms = vectorizer.get_feature_names_out() 

# What dose the vectorizer.fit_transform(document_list) return ?
''' A sparse matrix denoting

                 term 0      term 1        term 2     ...
document 0    tf-idf(0,0)  tf-idf(0,1)   tf-idf(0,2)    
document 1    tf-idf(1,0)  tf-idf(1,1)   tf-idf(1,2) 
document 2    tf-idf(2,0)  tf-idf(2,1)   tf-idf(2,2) 
   .
   .
   .

'''

# What is a sparse martix ?
'''
A sparse matrix in python is a matrix whose elements are mostly zero and thus are eliminated by python, 
one can use Matrix[a,b] to access its elements. A matrix that is mnot sparse is a dense matrix

dense = <sparse>.todense() : transform a sparse matrix to dense form
2Dlist = <dense>.tolist()  : transform a dense matrix to 2Dlist form
print(sparse) : can print the sparse matrix

'''

# whod dose the sklearn calculated the tf-idf ?
# https://www.analyticsvidhya.com/blog/2021/11/how-sklearns-tfidfvectorizer-calculates-tf-idf-values/
'''
Let : D be the total document list as 'corpus'
      d be one specific document in D as 'This is the first document.'
      t be one specific term that has appeared in D as 'This'
      n = Total number of documents available
      t = term for which idf value has to be calculated
      df(t) = Number of documents in which the term t appears
 
tf(t,d) = Number of times a term ‘t’ appears in a document d

idf(t,D) = ln[ (1+n) / ( 1 + df(t) ) ] + 1    (default i.e smooth_idf = True)
idf(t,D) = ln[ n / df(t) ] + 1                (when smooth_idf = False)

tf-idf(t,d,D) = tf(t,d)*idf(t,D)

sklearn than normalize the tf-idfs for each line in the sparse matrix:

                                 term 0      term 1        term 2     ...
                document 0    tf-idf(0,0)  tf-idf(0,1)   tf-idf(0,2)    
                document 1    tf-idf(1,0)  tf-idf(1,1)   tf-idf(1,2) 
                document 2    tf-idf(2,0)  tf-idf(2,1)   tf-idf(2,2) 
                   .
                   .
                   .
 
''';

In [ ]:
# Define a function, receiving a dataframe in the form as cbs_date_to_word and returns a dataframe recording the tf-idfs 
# of each appearing terms as a time sequence

'''   
                                    term 0      term 1        term 2     ...
                        day 0    tf-idf(0,0)  tf-idf(0,1)   tf-idf(0,2)  
                        day 1    tf-idf(1,0)  tf-idf(1,1)   tf-idf(1,2)  
                        day 2    tf-idf(2,0)  tf-idf(2,1)   tf-idf(2,2)  
                           .
                           .
                           .

'''

from analysis_utils import track_tfidf
    

In [ ]:
# Get statistical results
cbs_tfidf1 = track_tfidf( cbs_list1 )
cbs_tfidf_withna1 = cbs_tfidf1.copy()
cbs_tfidf1 = cbs_tfidf1.fillna(0)

In [ ]:
cbs_tfidf2 = track_tfidf( cbs_list2 )
cbs_tfidf_withna2 = cbs_tfidf2.copy()
cbs_tfidf2 = cbs_tfidf2.fillna(0)

In [ ]:
cbs_tfidf3 = track_tfidf( cbs_list3 )
cbs_tfidf_withna3 = cbs_tfidf3.copy()
cbs_tfidf3 = cbs_tfidf3.fillna(0)

In [ ]:
cbs_tfidf4 = track_tfidf( cbs_list4 )
cbs_tfidf_withna4 = cbs_tfidf4.copy()
cbs_tfidf4 = cbs_tfidf4.fillna(0)

In [ ]:
if save_data :
    cbs_tfidf1.to_csv('./Data/CBS_KDKA/TFIDF/cbs_tfidf1.csv', encoding = "utf_8_sig" ,  index = True)
    cbs_tfidf_withna1.to_csv('./Data/CBS_KDKA/TFIDF/cbs_tfidf_withna1.csv', encoding = "utf_8_sig" ,  index = True)
    
    cbs_tfidf2.to_csv('./Data/CBS_KDKA/TFIDF/cbs_tfidf2.csv', encoding = "utf_8_sig" ,  index = True)
    cbs_tfidf_withna2.to_csv('./Data/CBS_KDKA/TFIDF/cbs_tfidf_withna2.csv', encoding = "utf_8_sig" ,  index = True)
    
    cbs_tfidf3.to_csv('./Data/CBS_KDKA/TFIDF/cbs_tfidf3.csv', encoding = "utf_8_sig" ,  index = True)
    cbs_tfidf_withna3.to_csv('./Data/CBS_KDKA/TFIDF/cbs_tfidf_withna3.csv', encoding = "utf_8_sig" ,  index = True)
    
    cbs_tfidf4.to_csv('./Data/CBS_KDKA/TFIDF/cbs_tfidf4.csv', encoding = "utf_8_sig" ,  index = True)
    cbs_tfidf_withna4.to_csv('./Data/CBS_KDKA/TFIDF/cbs_tfidf_withna4.csv', encoding = "utf_8_sig" ,  index = True)

In [ ]:
cbs_tfidf3

## Visualization

### Data & Functions 

In [ ]:
# Read Data
cbs_tfidf1 = pd.read_csv('./Data/CBS_KDKA/TFIDF/cbs_tfidf1.csv', encoding = "utf_8_sig")
cbs_tfidf_withna1 = pd.read_csv('./Data/CBS_KDKA/TFIDF/cbs_tfidf_withna1.csv', encoding = "utf_8_sig")

cbs_tfidf2 = pd.read_csv('./Data/CBS_KDKA/TFIDF/cbs_tfidf2.csv', encoding = "utf_8_sig")
cbs_tfidf_withna2 = pd.read_csv('./Data/CBS_KDKA/TFIDF/cbs_tfidf_withna2.csv', encoding = "utf_8_sig")

cbs_tfidf3 = pd.read_csv('./Data/CBS_KDKA/TFIDF/cbs_tfidf3.csv', encoding = "utf_8_sig")
cbs_tfidf_withna3 = pd.read_csv('./Data/CBS_KDKA/TFIDF/cbs_tfidf_withna3.csv', encoding = "utf_8_sig")

cbs_tfidf4 = pd.read_csv('./Data/CBS_KDKA/TFIDF/cbs_tfidf4.csv', encoding = "utf_8_sig")
cbs_tfidf_withna4 = pd.read_csv('./Data/CBS_KDKA/TFIDF/cbs_tfidf_withna4.csv', encoding = "utf_8_sig")

dflst = [ cbs_tfidf1, cbs_tfidf_withna1,
          cbs_tfidf2, cbs_tfidf_withna2,
          cbs_tfidf3, cbs_tfidf_withna3,
          cbs_tfidf4, cbs_tfidf_withna4 ]

for df in dflst:
    df.set_index('Date', inplace = True)
    
days = cbs_tfidf1.index

In [ ]:
if save_data :
    cbs_tfidf_withna1.mean(axis=1).to_csv('./Data/CBS_KDKA/TFIDF/cbs_dailyav_tfidf1.csv', encoding = "utf_8_sig" ,  index = True)
    cbs_tfidf_withna2.mean(axis=1).to_csv('./Data/CBS_KDKA/TFIDF/cbs_dailyav_tfidf2.csv', encoding = "utf_8_sig" ,  index = True)
    cbs_tfidf_withna3.mean(axis=1).to_csv('./Data/CBS_KDKA/TFIDF/cbs_dailyav_tfidf3.csv', encoding = "utf_8_sig" ,  index = True) 
    cbs_tfidf_withna4.mean(axis=1).to_csv('./Data/CBS_KDKA/TFIDF/cbs_dailyav_tfidf4.csv', encoding = "utf_8_sig" ,  index = True)
  

In [ ]:
# Plot the tf-idf trendings for top 100 words with a maximum numbers of non-zero values
def trendingheatmap(df,topNumber):
    top_index = df.mean().sort_values(ascending = False)[:topNumber].index
    top_statistics = df.loc[:,top_index]
    maxVal = top_statistics.max().max()
    minVal = top_statistics.min().min()
    fig = plt.figure(figsize = (12,12))
    sns.heatmap(top_statistics.T, center = (maxVal+minVal)/2 )
    plt.show() 
    return fig


# Average tf-idfs for each term
def plot_wordsAverage_tfidf( df, topNumber = 50 ):
    average = df.mean().sort_values( ascending = False )
    
    # Parameters
    labelsize = 20
    titlesize = 25
    ticksize = 20
    tickrotation = 90
    
    fig = plt.figure( figsize = (20, 10) ) 
    ser = average[:topNumber]

    x = np.arange(len(ser))
    bar_width = 0.5

    # Plot bar chart
    plt.rc('axes', axisbelow=True) # grid behind plot
    plt.grid(linestyle = '-', linewidth = 1)
    plt.bar(x , ser , label = 'Standardized Average TF-IDFs' , 
            width = bar_width , color = (0, 53/255, 102/255))


    # Add title & ticks
    plt.title('Word-wise Average TF-IDFs: Top %d'%topNumber, fontproperties = 'Times New Roman', 
              fontsize = titlesize, fontweight="bold");
    plt.xticks(x , ser.index, fontproperties = 'Times New Roman', fontsize = ticksize, 
               rotation = tickrotation, fontweight="bold");
    plt.yticks(fontproperties = 'Times New Roman', size = ticksize, fontweight="bold");

    return fig


# Average tf-idfs for each day
def plot_dailyAverage_tfidf( df ):

    # daily average tf-idf
    daily_average = df.apply(lambda x: x.mean() , axis=1 )

    # plot trending
    n = len(daily_average.index)
    daily_index = np.arange(n)
    fig = plt.figure( figsize = (16, 10) ) 

    plt.grid(linestyle = '-', linewidth = 1.5)
    plt.ylim(0.0005, daily_average.max()+0.002) 
    plt.plot ( daily_index , daily_average , 
              linewidth = 2 , linestyle = '-' , color = (0, 95/255, 115/255)) 

    plt.title('Local News Daily Average TF-IDF Trending ( Standardized )', 
              fontproperties = 'Times New Roman' , 
              fontsize = 20, fontweight="bold");
    
    plt.xticks(daily_index[0:n:5] , daily_average.index[0:n:5] , 
               fontproperties = 'Times New Roman' , fontsize = 15, 
               rotation = 60, fontweight="bold");
    
    plt.yticks(fontproperties = 'Times New Roman', size = 15, fontweight="bold");
    return fig

### Visualization

In [ ]:
fig1 = trendingheatmap(cbs_tfidf4,20)
fig2 = trendingheatmap(cbs_tfidf4,50)

if save_plot :
        fig1.savefig(r'./Plot/CBS_KDKA/cbs_tfidf_heapmap_top20.jpg', bbox_inches = 'tight')
        fig2.savefig(r'./Plot/CBS_KDKA/cbs_tfidf_heapmap_top50.jpg', bbox_inches = 'tight')

In [ ]:
fig3 = trendingheatmap(cbs_tfidf4,100)
fig4 = trendingheatmap(cbs_tfidf4,500)

if save_plot :
        fig3.savefig(r'./Plot/CBS_KDKA/cbs_tfidf_heapmap_top100.jpg', bbox_inches = 'tight')
        fig4.savefig(r'./Plot/CBS_KDKA/cbs_tfidf_heapmap_top500.jpg', bbox_inches = 'tight')

Selected words with generally deeper colors are more important (higher tf-idf) in the figures presented above. Terms related to "covid", "coronavirus" , "health" and "cases" seem to be the key-terms within the period of 2020/3/10 - 2020/8/9 in Pittsburgh according to the local news from CBS KDKA

In [ ]:
# Average tf-idfs for each term
def plot_wordsAverage_tfidf( df, topNumber = 50 ):
    average = df.mean().sort_values( ascending = False )
    
    # Parameters
    labelsize = 20
    titlesize = 25
    ticksize = 20
    tickrotation = 90
    
    fig = plt.figure( figsize = (20, 10) ) 
    ser = average[:topNumber]

    x = np.arange(len(ser))
    bar_width = 0.5

    # Plot bar chart
    plt.rc('axes', axisbelow=True) # grid behind plot
    plt.grid(linestyle = '-', linewidth = 1)
    plt.bar(x , ser , label = 'Standardized Average TF-IDFs' , 
            width = bar_width , color = (0, 53/255, 102/255))


    # Add title & ticks
    plt.title('Local News Word-wise Average TF-IDFs: Top %d'%topNumber, fontproperties = 'Times New Roman', 
              fontsize = titlesize, fontweight="bold");
    plt.xticks(x , ser.index, fontproperties = 'Times New Roman', fontsize = ticksize, 
               rotation = tickrotation, fontweight="bold");
    plt.yticks(fontproperties = 'Times New Roman', size = ticksize, fontweight="bold");

    return fig
      

In [ ]:
fig1 = plot_wordsAverage_tfidf( cbs_tfidf1 )
fig2 = plot_wordsAverage_tfidf( cbs_tfidf2 )

In [ ]:
fig3 = plot_wordsAverage_tfidf( cbs_tfidf3 )
fig4 = plot_wordsAverage_tfidf( cbs_tfidf4 )

In [ ]:
if save_plot :
        fig1.savefig(r'./Plot/CBS_KDKA/cbs_WordwiseAverage_tfidf_1.jpg', bbox_inches = 'tight')
        fig2.savefig(r'./Plot/CBS_KDKA/cbs_WordwiseAverage_tfidf_2.jpg', bbox_inches = 'tight')
        fig3.savefig(r'./Plot/CBS_KDKA/cbs_WordwiseAverage_tfidf_3.jpg', bbox_inches = 'tight')
        fig4.savefig(r'./Plot/CBS_KDKA/cbs_WordwiseAverage_tfidf_4.jpg', bbox_inches = 'tight')

In [ ]:
Fig1 = plot_dailyAverage_tfidf( cbs_tfidf_withna1 )
Fig2 = plot_dailyAverage_tfidf( cbs_tfidf_withna2 )

In [ ]:
Fig3 = plot_dailyAverage_tfidf( cbs_tfidf_withna3 )
Fig4 = plot_dailyAverage_tfidf( cbs_tfidf_withna4 )

In [ ]:
if save_plot :
    Fig1.savefig(r'./Plot/CBS_KDKA/cbs_DailyAverage_tfidf_1.jpg', bbox_inches = 'tight')
    Fig2.savefig(r'./Plot/CBS_KDKA/cbs_DailyAverage_tfidf_2.jpg', bbox_inches = 'tight')
    Fig3.savefig(r'./Plot/CBS_KDKA/cbs_DailyAverage_tfidf_3.jpg', bbox_inches = 'tight')
    Fig4.savefig(r'./Plot/CBS_KDKA/cbs_DailyAverage_tfidf_4.jpg', bbox_inches = 'tight')

# Topic Analysis LDA

In [ ]:
# sklearn
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import GridSearchCV

# gensim
import gensim
import gensim.corpora as corpora
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel, LdaModel

'''
CountVectorizer : 

The use of CountVectorizer is similar as TfidfVectorizer, instead, it returns
a sparse representation of the word counts rather than tf-idfs

similarly, use : 

vectorizer = CountVectorizer() to construct the model

X = vectorizer.fit_transform(documnet_list) to fit the model

vectorizer.get_feature_names_out() to get the feature



                                  term 0         term 1           term 2     ...
                document 0    wordcount(0,0)  wordcount(0,1)   wordcount(0,2)    
       X =      document 1    wordcount(1,0)  wordcount(1,1)   wordcount(1,2) 
                document 2    wordcount(2,0)  wordcount(2,1)   wordcount(2,2) 
                   .
                   .
                   .
''' ;

In [ ]:
# Read data
cbs       = pd.read_csv('./Data/CBS_KDKA/cbs_clean.csv', encoding = "utf_8_sig")
cbs_list1 = pd.read_csv('./Data/CBS_KDKA/cbs_title.csv', encoding = "utf_8_sig")
cbs_list2 = pd.read_csv('./Data/CBS_KDKA/cbs_keywords.csv', encoding = "utf_8_sig")
cbs_list3 = pd.read_csv('./Data/CBS_KDKA/cbs_content.csv', encoding = "utf_8_sig")
cbs_list4 = pd.read_csv('./Data/CBS_KDKA/cbs_selectedwords.csv', encoding = "utf_8_sig")

cbs_list1.set_index('Date', inplace = True) # Title
cbs_list2.set_index('Date', inplace = True) # Keywords
cbs_list3.set_index('Date', inplace = True) # Content
cbs_list4.set_index('Date', inplace = True) # Selectedwords

days = list(cbs_list1.index)

cbs_total_wordcounts1 = count_words( cbs_all_words1 )
cbs_total_wordcounts2 = count_words( cbs_all_words2 )
cbs_total_wordcounts3 = count_words( cbs_all_words3 )
cbs_total_wordcounts4 = count_words( cbs_all_words4 )

display(cbs_total_wordcounts1.iloc[:25].T)
display(cbs_total_wordcounts2.iloc[:15].T)
display(cbs_total_wordcounts3.iloc[:20].T)
display(cbs_total_wordcounts4.iloc[:25].T)

# Get (daily) document set
documents1 = list(cbs_list1.iloc[:,0])
documents2 = list(cbs_list2.iloc[:,0])
documents3 = list(cbs_list3.iloc[:,0])
documents4 = list(cbs_list4.iloc[:,0])

In [ ]:
# Define the model 
n_keyterms = 100  # Number of key terms selected
n_min = 10        # Ignore terms that have a document frequency strictly lower than the given threshold
n_max = 280        # Ignore terms that have a document frequency strictly higher than the given threshold

tf_vectorizer = CountVectorizer(max_features = n_keyterms , 
                                min_df = n_min , max_df = n_max )
tf = tf_vectorizer.fit_transform( documents4 )
keyterms = tf_vectorizer.get_feature_names_out()

In [ ]:
display(keyterms)
display(len(keyterms))

## Parameter Chosen

### Best Topic Numbers

In [ ]:
# Note
# dictionary and corpus in gensim
'''
# gensim
import gensim
import gensim.corpora as corpora
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel, LdaModel

# Suppose the given documents are in the following form

                    docs = [ 'doc1',
                             'doc2',
                             'doc3',
                               ...
                             'docn'  ]

# To use gensim to start the text analysis:


# Step 0: import module

  from gensim.corpora import Dictionary


# Step 1: get a words Distionary according to the given documents docs

  ## 1.1: Convert docs to a 2D list
          docs = [ doc.split(" ") for doc in docs ]
          
  ## 1.2: Create the words dictionary
          dictionary = Dictionary(docs) # Now docs is a 2D list
          
  ## 1.3: Use dictionary.filter_extremes( no_below = N , no_above = p )
  You can filterout words that occur less than N documents, or more than p percent of the documents.
  the new dictionary will replace the old one without assign the new to the old.

  ## 1.4: Use dictionary.id2token, namely id2word in the official documents
  You can get the Python dictionary form of dictionary as the following form
   
  temp = dictionary[0]  # This is only to "load" the dictionary.
  dictionary.id2token = {'Saudis': 0, 'The': 1, 'a': 2, 'acknowledge': 3, 'are': 4, 
               'preparing': 5, 'report': 6, 'that': 7, 'will': 8, 'Jamal': 9, 
               "Khashoggi's": 10, 'Saudi': 11, 'an': 12, 'death': 13, 
               'journalist': 14, 'of': 15, 'result': 16, 'the': 17, 'was': 18, 
               'intended': 19, 'interrogation': 20, 'lead': 21, 'one': 22, 
               'to': 23, 'went': 24, 'wrong,': 25, 'Turkey,': 26, 'abduction': 27, 
               'according': 28, 'from': 29, 'his': 30, 'sources.': 31, 'two': 32}
  
   or you can use dictionary directoly as id2word: id2word = dictionary

# Step 2: ceate Bag-of-words corpus using the following code
   
   corpus = [dictionary.doc2bow(doc) for doc in docs] # Now docs is a 2D list

   The corpus is a list of lists whose elements are 2D tuples with the following meaning
   ( Word index , Number of Appearence ), the inner lists represent the documents doc

                                   [  [(),...,()],
                                      [(),...,()],
                                      [(),...,()],
                                      ...
                                      [(),...,()]   ]

''';

In [ ]:
# Find proper topic numbers by calculating the perplexity and the coherence value
# https://investigate.ai/text-analysis/choosing-the-right-number-of-topics-for-a-scikit-learn-topic-model/#Using-GridSearchCV-to-pick-the-best-number-of-topics
# Concept, People and Places => Using GridSearchCV
# https://www.bilibili.com/video/BV1LQ4y1Q7xv?spm_id_from=333.999.0.0&vd_source=ff901644057cda4596a72384c16c4fb4
# lower perplexity, higher log-likelyhood, higher coherence value refers to a clearer topic selection


def CreateCorpusAndDic( docs , no_below , no_above ):
    # Return docslst, dictionary, corpus
    
    '''
                    docs = [ 'doc1',
                             'doc2',
                             'doc3',
                               ...
                             'docn'  ]  
                      
                    no_below = N 
                    no_above = p     
                    
                    filterout words that occur less than N documents, 
                    or more than p percent of the documents.
                    
                             '''
    docs = [ doc.split(" ") for doc in docs ]
    dictionary = Dictionary(docs) 
    dictionary.filter_extremes( no_below = no_below , no_above = no_above )
    corpus = [dictionary.doc2bow(doc) for doc in docs] 

    return dictionary, corpus


##################################################################################################

def compute_coherence_values(docs, no_below=20, no_above=0.5, start=2, end=30, step=1):
    # https://www.kaggle.com/code/sahilmahajan0/topic-modeling-using-lda/notebook?scriptVersionId=44109636
    # https://aistudio.baidu.com/aistudio/projectdetail/408063
    
    """
    Compute c_v coherence for various number of topics

    Parameters:
    ----------
    dictionary : Gensim dictionary
    corpus     : Gensim corpus
    docs       : List of input texts  docs = [ 'doc1', 'doc2', 'doc3', ...'docn' ] 
    end        : Max num of topics

    Returns:
    -------
    model_list : List of LDA topic models
    coherence_values : Coherence values corresponding to the LDA model with respective number of topics
    
    """
    coherence_values = []
    topic_nums = range(start, end+1, step)
    dictionary, corpus = CreateCorpusAndDic( docs , no_below , no_above )
    docs = [ doc.split(" ") for doc in docs ]
    temp = dictionary[0]  # This is only to "load" the dictionary.
    id2word = dictionary.id2token
    
    for topic_num in tqdm(topic_nums):
        
        ldamodel = gensim.models.ldamodel.LdaModel(corpus=corpus, 
                                                    num_topics=topic_num,
                                                    alpha=50/topic_num,    # alpha
                                                    eta=0.1,       # beta
                                                    chunksize=500, # Number of documents to be used in each training chunk
                                                    passes=10,     # Number of passes through the corpus during training.
                                                    id2word=id2word,
                                                    iterations=1000,
                                                    eval_every=None)

        # eval_every = None: Don't evaluate model perplexity, takes too much time.     

        coherencemodel = CoherenceModel(model = ldamodel, texts = docs, 
                                            dictionary = dictionary, coherence = 'c_v')

        coherence_values.append(coherencemodel.get_coherence())

    return coherence_values



##################################################################################################
def compute_Perplexity(docs, n_min, n_max, n_keyterms = 100, start=2, end=30, step=1):
 
    # n_keyterms   # Number of key terms selected
    # n_min        # Ignore terms that have a document frequency strictly lower than the given threshold
    # n_max        # Ignore terms that have a document frequency strictly higher than the given threshold

    
    # Add addtional Stopwords
    stopwords = ['county','pennsylvania','virginia','additional','ohio','bringing',
             'week','year','fall','countywide','area','state','report','statewide',
            'amid', 'pittsburgh','covid','district','announces','west','local','center',
            'day','phase','virus','plan','beaver','pandemic','north','washington','link',
            'july','april','announced','percent','reporting','probable','month','thing','lot',
            'hour']
    
    topic_nums = range(start, end+1, step)
    plexs = []
    
    for topic_num in tqdm(topic_nums):
        
        tf_vectorizer = CountVectorizer(max_features = n_keyterms , 
                                        min_df = n_min , max_df = n_max,  stop_words=stopwords )
        tf = tf_vectorizer.fit_transform( docs )

        lda_sample = LatentDirichletAllocation(n_components = topic_num,
                                        doc_topic_prior  = 50/topic_num,    # alpha
                                        topic_word_prior = 0.1,   # beta
                                        learning_method='online',
                                        max_iter = 1000)
        lda_sample.fit(tf)
        plexs.append(lda_sample.perplexity(tf))   # Calculate approximate perplexity for data X.
    
    return plexs


##################################################################################################

# plot perplexities
def plotScores(coherence_values, plexs):
    
    TitleSize = 23
    LabelSize = 23
    TickSize  = 20
    topic_nums = range(2,len(coherence_values)+2)

    fig = plt.figure( figsize = (20, 10) ) 
    
    # plot coherence_values
    plt.subplot(1,2,1)
    plt.title('KDKA News: Topic Number - Coherence Value', fontproperties = 'Times New Roman' , 
          fontsize = TitleSize, fontweight="bold");
    plt.grid(linestyle = '-', linewidth = 1.5)
    plt.plot ( topic_nums , coherence_values , linewidth = 2 , linestyle = '-' , color = (0, 95/255, 115/255)) 
    plt.xticks( topic_nums, fontproperties = 'Times New Roman' , fontsize = TickSize, fontweight = "bold");
    plt.yticks( fontproperties = 'Times New Roman' , fontsize = TickSize, fontweight = "bold");
    plt.ylabel('Coherence Value', fontproperties = 'Times New Roman' , fontsize = LabelSize, fontweight="bold")
    plt.xlabel('Topic Number'    , fontproperties = 'Times New Roman' , fontsize = LabelSize, fontweight="bold")
    
    # plot plexs
    plt.subplot(1,2,2)
    plt.title('KDKA News: Topic Number - Perplexity', fontproperties = 'Times New Roman' , 
          fontsize = TitleSize, fontweight="bold");
    plt.grid(linestyle = '-', linewidth = 1.5)
    plt.plot ( topic_nums , np.array(plexs), linewidth = 2 , linestyle = '-' , color = (0, 95/255, 115/255)) 
    plt.xticks( topic_nums, fontproperties = 'Times New Roman' , fontsize = TickSize, fontweight = "bold");
    plt.yticks( fontproperties = 'Times New Roman' , fontsize = TickSize, fontweight = "bold");
    plt.ylabel('Perplexity', fontproperties = 'Times New Roman' , fontsize = LabelSize, fontweight="bold")
    plt.xlabel('Topic Number'    , fontproperties = 'Times New Roman' , fontsize = LabelSize, fontweight="bold")
    
    return fig

In [ ]:
# Document 1
coherence_values1 = compute_coherence_values(documents1, no_below=10, no_above=0.8, end=20)
plexs1 = compute_Perplexity(documents1, n_min=10, n_max=0.8, n_keyterms = 100, end=20)

In [ ]:
fig1 = plotScores(coherence_values1, plexs1)
if save_plot :
       fig1.savefig(r'./Plot/CBS_KDKA/cbs_LDA_topicnums1.jpg', bbox_inches = 'tight')

documents 1 : 3 8

In [ ]:
# Document 2
coherence_values2 = compute_coherence_values(documents2, no_below=10, no_above=0.8, end=20)
plexs2 = compute_Perplexity(documents2, n_min=10, n_max=0.8, n_keyterms = 100, end=20)

In [ ]:
fig2 = plotScores(coherence_values2, plexs2)
if save_plot :
       fig2.savefig(r'./Plot/CBS_KDKA/cbs_LDA_topicnums2.jpg', bbox_inches = 'tight')

documents 2 : 5 6 7 8

In [ ]:
# Document 3
coherence_values3 = compute_coherence_values(documents3, no_below=10, no_above=0.8, end=20)
plexs3 = compute_Perplexity(documents3, n_min=10, n_max=0.8, n_keyterms = 100, end=20)

In [ ]:
fig3 = plotScores(coherence_values3, plexs3)
if save_plot :
       fig3.savefig(r'./Plot/CBS_KDKA/cbs_LDA_topicnums3.jpg', bbox_inches = 'tight')

documents 3 : 5 6 8 9 12

In [ ]:
# Document 4
coherence_values4 = compute_coherence_values(documents4, no_below=10, no_above=0.8, end=20)
plexs4 = compute_Perplexity(documents4, n_min=10, n_max=0.8, n_keyterms = 100, end=20)

In [ ]:
fig4 = plotScores(coherence_values4, plexs4)
if save_plot :
       fig4.savefig(r'./Plot/CBS_KDKA/cbs_LDA_topicnums4.jpg', bbox_inches = 'tight')

documents 4 : 4 7

documents 1 : 3 8

documents 2 : 5 6 7 8

documents 3 : 5 6 8 9 12

documents 4 : 4 7

### Other Parameters : GridSearchCV

In [ ]:
# from sklearn.decomposition import LatentDirichletAllocation
# from sklearn.model_selection import GridSearchCV

In [ ]:
# Add addtional Stopwords
stopwords = ['county','pennsylvania','virginia','additional','ohio','bringing',
             'week','year','fall','countywide','area','state','report','statewide',
            'amid', 'pittsburgh','covid','district','announces','west','local','center',
            'day','phase','virus','plan','beaver','pandemic','north','washington','link',
            'july','april','announced','percent','reporting','probable','month','thing','lot',
            'hour','hill','penn','city','bull','wear','wearing','staff','including','told',
            'august','decision','group','bar','high','stand','increase','june','needed','start',
            'today','feel','pennsylvanian','team','entire']

#### Documents 1

In [ ]:
# Documents 1
documents = documents1

In [ ]:
# Prepare vectorized document
n_keyterms = 50    # Number of key terms selected
n_min = 10         # Ignore terms that have a document frequency strictly lower than the given threshold
n_max = 0.8        # Ignore terms that have a document frequency strictly higher than the given threshold

tf_vectorizer = CountVectorizer(max_features = n_keyterms , 
                                min_df = n_min , max_df = n_max , stop_words=stopwords )
vectorized_documents = tf_vectorizer.fit_transform( documents )
keyterms1 = tf_vectorizer.get_feature_names_out() # Get keyterms

In [ ]:
%%time
# Search Paraeters
topic_nums = [3,8]
alpha = [50/x for x in topic_nums]

search_params = { 'n_components'    : topic_nums,
                  'learning_decay'  : [.5, .7, .6, .7],
                  'doc_topic_prior' : alpha,                    # alpha 50/n_topics
                  'topic_word_prior': [0.01, 0.05, 0.1, 0.15]}  # beta 

# Set up LDA with the options we'll keep static
model = LatentDirichletAllocation(learning_method='online',max_iter=500)
# Init Grid Search Class
# Use verbose>0 to visualize the process, can be seen in he terminal window
gridsearch = GridSearchCV(model, param_grid = search_params, n_jobs = -1, verbose=2) 
# Do the Grid Search
gridsearch.fit(vectorized_documents)

In [ ]:
# Get best Model
best_lda1 = gridsearch.best_estimator_

# What did we find?
print("Documents1, Best Model's Params: ", gridsearch.best_params_)
print("Documents1, Best Log Likelihood Score: ", gridsearch.best_score_) # higher, better
print("Documents1, Model Perplexity: ", best_lda1.perplexity(vectorized_documents))

#### Documents 2

In [ ]:
# Documents 2
documents = documents2

In [ ]:
# Prepare vectorized document
n_keyterms = 50    # Number of key terms selected
n_min = 10         # Ignore terms that have a document frequency strictly lower than the given threshold
n_max = 0.8         # Ignore terms that have a document frequency strictly higher than the given threshold

tf_vectorizer = CountVectorizer(max_features = n_keyterms , 
                                min_df = n_min , max_df = n_max , stop_words=stopwords  )
vectorized_documents = tf_vectorizer.fit_transform( documents )
keyterms2 = tf_vectorizer.get_feature_names_out() # Get keyterms

In [ ]:
%%time
# Search Paraeters
topic_nums = [5,6,7,8]
alpha = [50/x for x in topic_nums]

search_params = { 'n_components'    : topic_nums,
                  'learning_decay'  : [.5, .7, .6, .7],
                  'doc_topic_prior' : alpha,                   # alpha 50/n_topics
                  'topic_word_prior': [0.01, 0.05, 0.1, 0.15]} # beta

# Set up LDA with the options we'll keep static
model = LatentDirichletAllocation(learning_method='online',max_iter=500)
# Init Grid Search Class
gridsearch = GridSearchCV(model, param_grid = search_params, n_jobs = -1, verbose=2)
# Do the Grid Search
gridsearch.fit(vectorized_documents)

In [ ]:
# Get best Model
best_lda2 = gridsearch.best_estimator_

# What did we find?
print("Documents2, Best Model's Params: ", gridsearch.best_params_)
print("Documents2, Best Log Likelihood Score: ", gridsearch.best_score_)
print("Documents2, Model Perplexity: ", best_lda2.perplexity(vectorized_documents))

#### Documents 3

In [ ]:
# Documents 3
documents = documents3

In [ ]:
# Prepare vectorized document
n_keyterms = 50    # Number of key terms selected
n_min = 10         # Ignore terms that have a document frequency strictly lower than the given threshold
n_max = 0.8       # Ignore terms that have a document frequency strictly higher than the given threshold

tf_vectorizer = CountVectorizer(max_features = n_keyterms , 
                                min_df = n_min , max_df = n_max , stop_words=stopwords  )
vectorized_documents = tf_vectorizer.fit_transform( documents )
keyterms3 = tf_vectorizer.get_feature_names_out() # Get keyterms

In [ ]:
%%time
# Search Paraeters
topic_nums = [5,6,8,9,12]
alpha = [50/x for x in topic_nums]

search_params = { 'n_components'    : topic_nums,
                  'learning_decay'  : [.5, .7, .6, .7],
                  'doc_topic_prior' : alpha,                   # alpha 50/n_topics
                  'topic_word_prior': [0.01, 0.05, 0.1, 0.15]}  # beta

# Set up LDA with the options we'll keep static
model = LatentDirichletAllocation(learning_method='online',max_iter=500)
# Init Grid Search Class
gridsearch = GridSearchCV(model, param_grid = search_params, n_jobs = -1, verbose=2)
# Do the Grid Search
gridsearch.fit(vectorized_documents)

In [ ]:
# Get best Model
best_lda3 = gridsearch.best_estimator_

# What did we find?
print("Documents3, Best Model's Params: ", gridsearch.best_params_)
print("Documents3, Best Log Likelihood Score: ", gridsearch.best_score_)
print("Documents3, Model Perplexity: ", best_lda3.perplexity(vectorized_documents))

#### Documents 4

In [ ]:
# Documents 4
documents = documents4

In [ ]:
# Prepare vectorized document
n_keyterms = 50    # Number of key terms selected
n_min = 10         # Ignore terms that have a document frequency strictly lower than the given threshold
n_max = 0.8        # Ignore terms that have a document frequency strictly higher than the given threshold

tf_vectorizer = CountVectorizer(max_features = n_keyterms , 
                                min_df = n_min , max_df = n_max , stop_words=stopwords  )
vectorized_documents = tf_vectorizer.fit_transform( documents )
keyterms4 = tf_vectorizer.get_feature_names_out() # Get keyterms

In [ ]:
%%time
# Search Paraeters
topic_nums = [4,5,6,7]
alpha = [50/x for x in topic_nums]

search_params = { 'n_components'    : topic_nums,
                  'learning_decay'  : [.5, .7, .6, .7],
                  'doc_topic_prior' : alpha,                    # alpha 50/n_topics
                  'topic_word_prior': [0.01, 0.05, 0.1, 0.15]}  # beta

# Set up LDA with the options we'll keep static
model = LatentDirichletAllocation(learning_method='online',max_iter=500)
# Init Grid Search Class
gridsearch = GridSearchCV(model, param_grid = search_params, n_jobs = -1, verbose=2)
# Do the Grid Search
gridsearch.fit(vectorized_documents)

In [ ]:
# Get best Model
best_lda4 = gridsearch.best_estimator_

# What did we find?
print("Documents4, Best Model's Params: ", gridsearch.best_params_)
print("Documents4, Best Log Likelihood Score: ", gridsearch.best_score_)
print("Documents4, Model Perplexity: ", best_lda4.perplexity(vectorized_documents))

In [ ]:
# Save LDAModels
import pickle

LDAModels = [best_lda1, best_lda2, best_lda3, best_lda4]

if True :
    for lda in LDAModels :
        filename = './Data/CBS_KDKA/LDA/LDA%d.pkl'%(LDAModels.index(lda)+1)
        with open(filename, 'wb') as file:
            pickle.dump(lda, file)

In [ ]:
LDAModels

Documents1, Best Log Likelihood Score:  -2464.233026111187, Model Perplexity:  43.55261244973223

Documents2, Best Log Likelihood Score:  -280.6536127116992, Model Perplexity:  16.654093416900416

Documents3, Best Log Likelihood Score:  -15705.48205922542, Model Perplexity:  42.43833887701525

Documents4, Best Log Likelihood Score:  -2664.936777665152, Model Perplexity:  43.91184302808175

## LDA Model: Topic with Top Words

In [ ]:
# Notes --- Apply LDA
# https://stats.stackexchange.com/questions/59684/what-are-typical-values-to-use-for-alpha-and-beta-in-latent-dirichlet-allocation

'''
n_topics = 5 # Number of topics

lda = LatentDirichletAllocation(n_components = n_topics,
                                doc_topic_prior  = 50/n_topics,    # alpha
                                topic_word_prior = 0.1,   # beta
                                learning_method='batch',
                                learning_offset = 50.0, 
                                max_iter = 50)

lda.fit(tf) # tf is the sparse matrix from the CountVectorizer model

''';

### Additional Stopwords

In [ ]:
# Add addtional Stopwords
# Visualiztion
import pyLDAvis
import pyLDAvis.lda_model

stopwords = ['county','pennsylvania','virginia','additional','ohio','bringing',
             'week','year','fall','countywide','area','state','report','statewide',
            'amid', 'pittsburgh','covid','district','announces','west','local','center',
            'day','phase','virus','plan','beaver','pandemic','north','washington','link',
            'july','april','announced','percent','reporting','probable','month','thing','lot',
            'hour','hill','penn','city','bull','wear','wearing','staff','including','told',
            'august','decision','group','bar','high','stand','increase','june','needed','start',
            'today','feel','pennsylvanian','team','entire']

In [ ]:
# Load the saved model and data
import pickle
# Read data
cbs       = pd.read_csv('./Data/CBS_KDKA/cbs_clean.csv', encoding = "utf_8_sig")
cbs_list1 = pd.read_csv('./Data/CBS_KDKA/cbs_title.csv', encoding = "utf_8_sig")
cbs_list2 = pd.read_csv('./Data/CBS_KDKA/cbs_keywords.csv', encoding = "utf_8_sig")
cbs_list3 = pd.read_csv('./Data/CBS_KDKA/cbs_content.csv', encoding = "utf_8_sig")
cbs_list4 = pd.read_csv('./Data/CBS_KDKA/cbs_selectedwords.csv', encoding = "utf_8_sig")

cbs_list1.set_index('Date', inplace = True) # Title
cbs_list2.set_index('Date', inplace = True) # Keywords
cbs_list3.set_index('Date', inplace = True) # Content
cbs_list4.set_index('Date', inplace = True) # Selectedwords

days = list(cbs_list1.index)

# Get (daily) document set
documents1 = list(cbs_list1.iloc[:,0])
documents2 = list(cbs_list2.iloc[:,0])
documents3 = list(cbs_list3.iloc[:,0])
documents4 = list(cbs_list4.iloc[:,0])

documents = [documents1, documents2, documents3, documents4]

# Load the saved model
LDAModels = []
tfs = []
tf_vectorizers = []
keyterms = []
n_maxs = [0.8,0.8,0.8,0.8] #[219, 64, 1731, 649]

for i in range(4) :
    filename = './Data/CBS_KDKA/LDA/LDA%d.pkl'%(i+1)
    with open(filename, 'rb') as file:
        LDAModels.append(pickle.load(file))  

    # Prepare vectorized document
    n_keyterms = 50     # Number of key terms selected
    n_min = 10          # Ignore terms that have a document frequency strictly lower than the given threshold
    n_max = n_maxs[i]   # Ignore terms that have a document frequency strictly higher than the given threshold

    vectorizer = CountVectorizer(max_features = n_keyterms , 
                                    min_df = n_min , max_df = n_max , stop_words=stopwords  )
    tf = vectorizer.fit_transform( documents[i] )
    tf_vectorizers.append(vectorizer)
    tfs.append(tf)
    keyterms.append(vectorizer.get_feature_names_out()) # Get keyterms


In [ ]:
display(LDAModels)
display(tfs)
display(tf_vectorizers)
display(keyterms)

In [ ]:
# Model result
lda = LDAModels[0]

# model.components_ 
display( lda.components_ )

'''
model.components_[i, j] :（pseudocount）the number of times word j was assigned to topic i
model.components_ / model.components_.sum(axis=1)[:, np.newaxis] ： the proportion of word j in topic i
line i for topic, column j for key feature / term

pseudocount :
A pseudocount is an amount (not generally an integer, despite its name) added to the number of observed
cases in order to change the expected probability in a model of those data, when not known to be zero.
https://www.zhihu.com/question/23405711

'''

# model.n_features_in_
display( lda.n_features_in_ )   # Number of features / keyterms

# model.perplexity(tf)
# tf is the sparse matrix from the CountVectorizer model
display( lda.perplexity(tf) )   # calculate the model perplexity

# model.transform(tf)
# tf is the sparse matrix from the CountVectorizer model
display( lda.transform(tf) )   # The topics associated with the documents


'''  model.transform(tf) = 

               ( 2Darray )      Topic 0         Topic 1        Topic 2      ....
                document 0    percent(0,0)   percent(0,1)   percent(0,2)
                document 1    percent(1,0)   percent(1,1)   percent(1,2)
                document 2    percent(2,0)   percent(2,1)   percent(2,2)
                .
                .
                .

'''

In [ ]:
# Define a function to print the top words in each topic
def print_top_words( model , feature_names , n_top_words ):
    
    '''
    Return a list of dictionaries, with key being the keyterm and value being the corresponding percentage
    '''
    
    tword = []
    percent_component = model.components_ / model.components_.sum(axis=1)[:, np.newaxis]
    for topic_id, topic in enumerate(percent_component):
        
        topic_w = { feature_names[i] : [round(topic[i],3),] for i in topic.argsort()[-1::-1][:n_top_words] }
        tword.append(topic_w)
        
        total_percent = sum([topic[i] for i in topic.argsort()[-1::-1][:n_top_words]])*100
        
        print('Topic #%d' % (topic_id+1), ':' , '%.2f'%total_percent , '%' )
        print(topic_w)
        print(' ')
    
    return tword


# Define a function to print the percent for top words in each topic
def top_words_percent( model , n_top_words ):
    percent = []
    percent_component = model.components_ / model.components_.sum(axis=1)[:, np.newaxis]
    for topic in percent_component:     
        total_percent = sum([topic[i] for i in topic.argsort()[-1::-1][:n_top_words]])
        percent.append(total_percent)
    return np.array(percent)


# Train New Model
def NewLDA( index , topicNum = -1, maxIter = -1, update = False ):
    # index is the document index - 1
    doc_topic_priors  = [6.25, 5.555555555555555, 4.545454545454546, 8.333333333333334]
    learning_decays   = [0.5, 0.5, 0.5, 0.5]
    topic_word_priors = [0.15, 0.1, 0.15, 0.15]
    topic_nums        = [3, 5, 5, 4]
    max_iters         = [500, 500, 500, 500]
    
    if topicNum > 0 :
        topic_nums[index] = topicNum
    if maxIter > 0 :
        max_iters [index] = maxIter
    
    lda = LatentDirichletAllocation(doc_topic_prior=doc_topic_priors[index], 
                                    learning_decay=learning_decays[index] ,
                                    learning_method='online', 
                                    max_iter=max_iters[index],
                                    n_components=topic_nums[index], 
                                    topic_word_prior=topic_word_priors[index])
    lda.fit(tfs[index])
    perp = lda.perplexity(tfs[index])
    score  = lda.score(tfs[index])
    
    print("Documents%d, Best Log Likelihood Score: "%(index+1),score)
    print("Documents%d, Model Perplexity: "%(index+1),perp)
    
    if update:
        filename = './Data/CBS_KDKA/LDA/adjusted_LDA%d.pkl'%(index+1)
        with open(filename, 'wb') as file:
            pickle.dump(lda, file)
            
        pyLDAvis.enable_notebook();
        pic = pyLDAvis.lda_model.prepare(lda, tfs[index], tf_vectorizers[index]);
        pyLDAvis.display(pic);
        pyLDAvis.save_html(pic,'./Data/CBS_KDKA/LDA/adjusted_cbs_lda_visual%d.html'%(index+1)); # open lda.html in the working dictionary

    return lda


### Documents 1

In [ ]:
# Notes --- Apply LDA
# https://stats.stackexchange.com/questions/59684/what-are-typical-values-to-use-for-alpha-and-beta-in-latent-dirichlet-allocation

'''
n_topics = 5 # Number of topics

lda = LatentDirichletAllocation(n_components = n_topics,
                                doc_topic_prior  = 50/n_topics,    # alpha
                                topic_word_prior = 0.1,   # beta
                                learning_method='batch',
                                learning_offset = 50.0, 
                                max_iter = 50)

lda.fit(tf) # tf is the sparse matrix from the CountVectorizer model

''';

In [ ]:
%%time
LDAModels[0] = NewLDA( 0 , topicNum = -1, maxIter = 5000, update = True );

In [ ]:
cbs_topic_dics1 = print_top_words( LDAModels[0] , keyterms[0] , n_top_words = 10 )

Topic1: Close & Open policies

Topic2: Medical, Test count & Safety 

Topic3: School & University

In [ ]:
if save_data :
    for topic in cbs_topic_dics1 :
        topic_index = 'Topic_%d' % (cbs_topic_dics1.index(topic)+1)
        pd.DataFrame(topic, index = [topic_index,]).to_csv('./Data/CBS_KDKA/LDA/Title/cbsTopicTopWords_'+ topic_index + '.csv', encoding = "utf_8_sig" ,  index = False)

### Documents 2

In [ ]:
%%time
LDAModels[1] = NewLDA( 1 , topicNum = -1, maxIter = 1000, update = True );

In [ ]:
cbs_topic_dics2 = print_top_words( LDAModels[1] , keyterms[1] , n_top_words = 10 )

In [ ]:
if save_data :
    for topic in cbs_topic_dics2 :
        topic_index = 'Topic_%d' % (cbs_topic_dics2.index(topic)+1)
        pd.DataFrame(topic, index = [topic_index,]).to_csv('./Data/CBS_KDKA/LDA/Keyword/cbsTopicTopWords_'+ topic_index + '.csv', encoding = "utf_8_sig" ,  index = False)

### Documents 3

In [ ]:
LDAModels[2] = NewLDA( 2 , topicNum = -1, maxIter = 5000, update = True );

In [ ]:
cbs_topic_dics3 = print_top_words( LDAModels[2] , keyterms[2] , n_top_words = 10 )

In [ ]:
if save_data :
    for topic in cbs_topic_dics3 :
        topic_index = 'Topic_%d' % (cbs_topic_dics3.index(topic)+1)
        pd.DataFrame(topic, index = [topic_index,]).to_csv('./Data/CBS_KDKA/LDA/Content/cbsTopicTopWords_'+ topic_index + '.csv', encoding = "utf_8_sig" ,  index = False)

### Documents 4

In [ ]:
%%time
LDAModels[3] = NewLDA( 3 , topicNum = -1, maxIter = 1000, update = True );

In [ ]:
cbs_topic_dics4 = print_top_words( LDAModels[3] , keyterms[3] , n_top_words = 10 )

Topic1: School & University Life

Topic2: Outbreak, Close & Gov Policy

Topic3: Test count & Work

Topic4: Reopen & Safety & Restaurant

In [ ]:
if save_data :
    for topic in cbs_topic_dics4 :
        topic_index = 'Topic_%d' % (cbs_topic_dics4.index(topic)+1)
        pd.DataFrame(topic, index = [topic_index,]).to_csv('./Data/CBS_KDKA/LDA/KeywordAndTitle/cbsTopicTopWords_'+ topic_index + '.csv', encoding = "utf_8_sig" ,  index = False)

In [ ]:
# Visualiztion
import pyLDAvis
import pyLDAvis.lda_model

for i in range(4):
    pyLDAvis.enable_notebook()
    pic = pyLDAvis.lda_model.prepare(LDAModels[i], tfs[i], tf_vectorizers[i])
    pyLDAvis.display(pic)
    pyLDAvis.save_html(pic,'./Data/CBS_KDKA/LDA/cbs_lda_visual%d.html'%(i+1)) # open lda.html in the working dictionary
    print('Finish %d'%(i+1))

## Topic Associations with the Documents

### Data

In [ ]:
# Load the saved model and data
import pickle

stopwords = ['county','pennsylvania','virginia','additional','ohio','bringing',
             'week','year','fall','countywide','area','state','report','statewide',
            'amid', 'pittsburgh','covid','district','announces','west','local','center',
            'day','phase','virus','plan','beaver','pandemic','north','washington','link',
            'july','april','announced','percent','reporting','probable','month','thing','lot',
            'hour','hill','penn','city','bull','wear','wearing','staff','including','told',
            'august','decision','group','bar','high','stand','increase','june','needed','start',
            'today','feel','pennsylvanian','team','entire']

# Read data
cbs       = pd.read_csv('./Data/CBS_KDKA/cbs_clean.csv', encoding = "utf_8_sig")
cbs_list1 = pd.read_csv('./Data/CBS_KDKA/cbs_title.csv', encoding = "utf_8_sig")
cbs_list2 = pd.read_csv('./Data/CBS_KDKA/cbs_keywords.csv', encoding = "utf_8_sig")
cbs_list3 = pd.read_csv('./Data/CBS_KDKA/cbs_content.csv', encoding = "utf_8_sig")
cbs_list4 = pd.read_csv('./Data/CBS_KDKA/cbs_selectedwords.csv', encoding = "utf_8_sig")

cbs_list1.set_index('Date', inplace = True) # Title
cbs_list2.set_index('Date', inplace = True) # Keywords
cbs_list3.set_index('Date', inplace = True) # Content
cbs_list4.set_index('Date', inplace = True) # Selectedwords

days = list(cbs_list1.index)

# Get (daily) document set
documents1 = list(cbs_list1.iloc[:,0])
documents2 = list(cbs_list2.iloc[:,0])
documents3 = list(cbs_list3.iloc[:,0])
documents4 = list(cbs_list4.iloc[:,0])

documents = [documents1, documents2, documents3, documents4]

# Load the saved model
LDAModels = []
tfs = []
tf_vectorizers = []
keyterms = []
n_maxs = [0.8,0.8,0.8,0.8] #[219, 64, 1731, 649]

for i in range(4) :
    filename = './Data/CBS_KDKA/LDA/adjusted_LDA%d.pkl'%(i+1)
    with open(filename, 'rb') as file:
        LDAModels.append(pickle.load(file))  

    # Prepare vectorized document
    n_keyterms = 50     # Number of key terms selected
    n_min = 10          # Ignore terms that have a document frequency strictly lower than the given threshold
    n_max = n_maxs[i]   # Ignore terms that have a document frequency strictly higher than the given threshold

    vectorizer = CountVectorizer(max_features = n_keyterms , 
                                    min_df = n_min , max_df = n_max , stop_words=stopwords  )
    tf = vectorizer.fit_transform( documents[i] )
    tf_vectorizers.append(vectorizer)
    tfs.append(tf)
    keyterms.append(vectorizer.get_feature_names_out()) # Get keyterms


In [ ]:
# The topics associated with the documents
def generateTopicComposition(df, model, tf, topicNum):
    days = list(df.index)
    topics = ['Topic %d'%(x+1) for x in range(topicNum)]
    dist = model.transform(tf)
    
    for day in tqdm(days) :
        for topic in topics:
            df.loc[day,topic] = dist[days.index(day) , topics.index(topic)]
            
    return df

In [ ]:
# The topics associated with the documents
cbs_list1 = generateTopicComposition(cbs_list1, LDAModels[0], tfs[0], 3)
cbs_list2 = generateTopicComposition(cbs_list2, LDAModels[1], tfs[1], 5)
cbs_list3 = generateTopicComposition(cbs_list3, LDAModels[2], tfs[2], 5)
cbs_list4 = generateTopicComposition(cbs_list4, LDAModels[3], tfs[3], 4)

In [ ]:
# Save the Data
if save_data :
    cbs_list1.to_csv('./Data/CBS_KDKA/LDA/cbs_DailyTopicPercent1.csv', encoding = "utf_8_sig" ,  index = True)
    cbs_list2.to_csv('./Data/CBS_KDKA/LDA/cbs_DailyTopicPercent2.csv', encoding = "utf_8_sig" ,  index = True)
    cbs_list3.to_csv('./Data/CBS_KDKA/LDA/cbs_DailyTopicPercent3.csv', encoding = "utf_8_sig" ,  index = True)
    cbs_list4.to_csv('./Data/CBS_KDKA/LDA/cbs_DailyTopicPercent4.csv', encoding = "utf_8_sig" ,  index = True)

### Visualization

In [ ]:
# Read data
cbs_dtp_list1 = pd.read_csv('./Data/CBS_KDKA/LDA/cbs_DailyTopicPercent1.csv', encoding = "utf_8_sig")
cbs_dtp_list2 = pd.read_csv('./Data/CBS_KDKA/LDA/cbs_DailyTopicPercent2.csv', encoding = "utf_8_sig")
cbs_dtp_list3 = pd.read_csv('./Data/CBS_KDKA/LDA/cbs_DailyTopicPercent3.csv', encoding = "utf_8_sig")
cbs_dtp_list4 = pd.read_csv('./Data/CBS_KDKA/LDA/cbs_DailyTopicPercent4.csv', encoding = "utf_8_sig")

cbs_dtp_list1.set_index('Date',inplace = True)
cbs_dtp_list2.set_index('Date',inplace = True)
cbs_dtp_list3.set_index('Date',inplace = True)
cbs_dtp_list4.set_index('Date',inplace = True)

In [ ]:
cbs_dtp_list1

In [ ]:
# Visualization 1
def drawTrend_heatmap(cbs_dtp_list,delta=0.1): 
    # delta is the row label coordinate
    topic_percent_data = cbs_dtp_list.loc[:,'Topic 1':]
    maxVal = topic_percent_data.max().max()
    minVal = topic_percent_data.min().min()
    n = len(cbs_dtp_list.index)
    daily_index = np.arange(n)
    topicNum = len(topic_percent_data.columns)
    
    fig = plt.figure(figsize = (20,5))
    sns.heatmap(topic_percent_data.T, center = (maxVal+minVal)/2, xticklabels=4, cmap = "rocket_r" )
    plt.yticks(np.arange(topicNum)+delta,topic_percent_data.T.index,
               fontproperties = 'Times New Roman', size = 15, fontweight="bold");
    plt.xticks(fontproperties = 'Times New Roman', size = 15, fontweight="bold");
    plt.xlabel('',fontproperties = 'Times New Roman',fontsize = 25,fontweight="bold")
    plt.show()
    return fig

In [ ]:
fig = drawTrend_heatmap(cbs_dtp_list1,0.3)
if save_plot :
       fig.savefig(r'./Plot/CBS_KDKA/topic_trending_heatmap1.jpg', bbox_inches = 'tight')

In [ ]:
fig = drawTrend_heatmap(cbs_dtp_list2,0.1)
if save_plot :
       fig.savefig(r'./Plot/CBS_KDKA/topic_trending_heatmap2.jpg', bbox_inches = 'tight')

In [ ]:
fig = drawTrend_heatmap(cbs_dtp_list3,0.1)
if save_plot :
       fig.savefig(r'./Plot/CBS_KDKA/topic_trending_heatmap3.jpg', bbox_inches = 'tight')

In [ ]:
fig = drawTrend_heatmap(cbs_dtp_list4,0.2)
if save_plot :
       fig.savefig(r'./Plot/CBS_KDKA/topic_trending_heatmap4.jpg', bbox_inches = 'tight')

In [ ]:
#  Visualization 2
def drawTrend(cbs_dtp_list,title_y_coordinate=0.55): 
    topic_percent_data = cbs_dtp_list.loc[:,'Topic 1':]
    maxVal = topic_percent_data.max().max()
    minVal = topic_percent_data.min().min()
    n = len(cbs_dtp_list.index)
    daily_index = np.arange(n)
    topicNum = len(topic_percent_data.columns)
    topics = ['Topic %d'%(x+1) for x in range(topicNum)]
    
    Colors_RGB = [ [155, 34, 38],
                 [202, 103, 2],
                 [10, 147, 150],
                 [0, 95, 115],
                 [0, 18, 25] ]

    Colors = list(np.array(Colors_RGB)/255)

    fig = plt.figure(figsize = (20,10))
    fig.text(0.09, title_y_coordinate, 'Percentage Topic Composition %', va='center', rotation='vertical', 
         fontproperties = 'Times New Roman',fontsize = 20,fontweight="bold")
    
    grid = plt.GridSpec(36, 10, wspace=1.5, hspace=1.5);
    
    startindex = list(np.arange(topicNum)*6)
    subplotindex = [0,4,6,10,12,16,18,22,24,28,30]
    
    
    # Subplots
    for i in startindex :
        plt.subplot(grid[i:i+4,:])
        plt.plot ( daily_index, 25*np.ones_like(daily_index) , linewidth = 2 ,
                   linestyle = '-.' , color = (141/255, 153/255, 174/255)  ) 
        h = plt.plot ( daily_index, topic_percent_data[topics[startindex.index(i)]]*100 , linewidth = 2 ,
                   linestyle = '-' , color = tuple(Colors[startindex.index(i)]) )

        plt.xticks([]) 
        plt.xlim(0,len(daily_index)) 
        plt.legend( handles = h, labels = [topics[startindex.index(i)]] , loc = 'upper right' , 
                   prop = {'weight':'bold','family':'Times New Roman'})

        plt.subplot(grid[i+4:i+6,:])
        ax = sns.heatmap(pd.DataFrame(topic_percent_data[topics[startindex.index(i)]]).T, 
                    center = (maxVal+minVal)/2, cbar = False, 
                    xticklabels = False, yticklabels = False)
        plt.xlabel('',fontproperties = 'Times New Roman',fontsize = 25,fontweight="bold")
       
        if startindex.index(i) == topicNum-1:
            ax.set_xticks(list(daily_index[0::3]))
            ax.set_xticklabels(list(topic_percent_data.index[list(daily_index[0::3])]))
            plt.xticks(fontproperties = 'Times New Roman', size = 15, fontweight="bold",rotation=90)
        
        
        
    return fig


In [ ]:
fig1 =  drawTrend(cbs_dtp_list1, title_y_coordinate = 0.65)

In [ ]:
fig2 =  drawTrend(cbs_dtp_list2, title_y_coordinate = 0.55)

In [ ]:
fig3 =  drawTrend(cbs_dtp_list3, title_y_coordinate = 0.55)

In [ ]:
fig4 =  drawTrend(cbs_dtp_list4, title_y_coordinate = 0.64)

In [ ]:
if save_plot :
    fig1.savefig(r'./Plot/CBS_KDKA/topic_trending1.jpg', bbox_inches = 'tight')
    fig2.savefig(r'./Plot/CBS_KDKA/topic_trending2.jpg', bbox_inches = 'tight')
    fig3.savefig(r'./Plot/CBS_KDKA/topic_trending3.jpg', bbox_inches = 'tight')
    fig4.savefig(r'./Plot/CBS_KDKA/topic_trending4.jpg', bbox_inches = 'tight')